# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imhs14/flyrank-ml-internship-starter-main/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal 1 — staleness (behind FlyRank's refresh flags):** FALSE. I checked days_since_last_update
against avg_position (correlation ≈ 0.07 — essentially none), and the freshness_tier bucket table
doesn't move in a consistent direction either. Two of the four tiers (31-90 and 181+) have under
200 rows each, against 20,000+ in the "0-30" tier, so those extremes aren't trustworthy anyway.
Staleness alone tells me nothing here — I'm dropping it from my rule.

**Signal 2 — CTR vs position (behind the needs_ctr_fix logic):** CONFIRMED. CTR falls cleanly and
monotonically as position gets worse: top_3 (1.48) → page_1 (0.65) → striking (0.32) →
page_3_5 (0.22) → deep (0.15). Every bucket has a solid sample size (1,300+ rows).

**My rule, in plain words:** A page deserves a CTR-fix review if it already gets real search demand
(impressions_90d ≥ 300) but its CTR sits well below what other pages at the same position tier
typically earn. That gap — ranking fine but not earning the clicks that ranking should produce — is
exactly what needs_ctr_fix is meant to catch, and it's the one signal I actually confirmed above.

**Reason code (one, applied to every flagged row):** `underperforming_ctr_for_position`

**Action label:** `improve` if the rule fires, `monitor` otherwise.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# find the repo root from wherever this kernel started
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /Users/Heerashanker/Downloads/Main/Coding/FlyRank-AI/flyrank-ml-internship-starter-main
Starter data found. You're ready.


In [13]:
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print("=== Signal 1: staleness (freshness_tier) vs avg_position ===")
print(df.groupby('freshness_tier')['avg_position'].agg(['mean', 'count']))
print("Correlation (days_since_last_update vs avg_position):", round(df['days_since_last_update'].corr(df['avg_position']), 3))
print()

print("=== Signal 2: position_tier vs ctr ===")
print(df.groupby('position_tier')['ctr'].agg(['mean', 'count']))

=== Signal 1: staleness (freshness_tier) vs avg_position ===
                     mean  count
freshness_tier                  
0-30            15.685166  20480
181+            11.325862    174
31-90           16.538286    175
91-180          17.901461   9171
Correlation (days_since_last_update vs avg_position): 0.07

=== Signal 2: position_tier vs ctr ===
                   mean  count
position_tier                 
deep           0.150212   1319
page_1         0.652467  11814
page_3_5       0.222484   7242
striking       0.323239   7304
top_3          1.483611   2321


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
peer_avg_ctr = df.groupby('position_tier')['ctr'].transform('mean')
visible = (df['impressions_90d'] >= 300).astype(int)
ctr_gap = (peer_avg_ctr - df['ctr']).clip(lower=0)  # how far below peers; 0 if already fine

df['baseline_action_score'] = visible * ctr_gap * np.log1p(df['impressions_90d'])
df['reason_code'] = 'underperforming_ctr_for_position'
df['action_label'] = np.where(df['baseline_action_score'] > 0, 'improve', 'monitor')
df['baseline_rank'] = df['baseline_action_score'].rank(method='first', ascending=False).astype(int)

out_cols = ['content_id', 'client_id', 'baseline_rank', 'baseline_action_score',
            'reason_code', 'action_label', 'position_tier', 'avg_position', 'ctr',
            'impressions_90d', 'freshness_tier']
out = df[out_cols].sort_values('baseline_rank')

import os, json
os.makedirs('work/outputs', exist_ok=True)
out.to_csv('work/outputs/baseline_action_score.csv', index=False)

metadata = {
    "rows": int(len(out)),
    "top_score": float(out['baseline_action_score'].max()),
    "median_score": float(out['baseline_action_score'].median()),
    "flagged_improve_count": int((df['action_label'] == 'improve').sum()),
    "reason_code": "underperforming_ctr_for_position",
}
with open('work/outputs/baseline_action_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Wrote {len(out)} rows. Improve: {metadata['flagged_improve_count']}, Monitor: {len(out) - metadata['flagged_improve_count']}")
out.head(10)

Wrote 30000 rows. Improve: 15319, Monitor: 14681


,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action_label,position_tier,avg_position,ctr,impressions_90d,freshness_tier
7678,content_8451fc6f034d,client_d029fa3a95,1,18.190613,underperforming_ctr_for_position,improve,top_3,2.3,0.03,272144,0-30
26844,content_8c19996aa890,client_4e07408562,2,17.524576,underperforming_ctr_for_position,improve,top_3,2.5,0.15,509252,0-30
3331,content_4a6607efcb46,client_6208ef0f77,3,17.330138,underperforming_ctr_for_position,improve,top_3,2.2,0.01,128068,91-180
16736,content_e12868d1f396,client_4e07408562,4,16.845255,underperforming_ctr_for_position,improve,top_3,2.9,0.07,149712,0-30
9412,content_8053a66bd6ac,client_19581e27de,5,15.260254,underperforming_ctr_for_position,improve,top_3,2.6,0.08,52687,91-180
7860,content_d225ec9f3d46,client_f369cb89fc,6,14.599610,underperforming_ctr_for_position,improve,top_3,0.7,0.05,26470,0-30
27107,content_0022a6b4290f,client_f369cb89fc,7,14.560919,underperforming_ctr_for_position,improve,top_3,1.2,0.07,29747,0-30
10893,content_6f81ccd92b64,client_19581e27de,8,14.498052,underperforming_ctr_for_position,improve,top_3,2.9,0.19,73675,91-180
20537,content_f4e210ee0c27,client_7f2253d7e2,9,14.404082,underperforming_ctr_for_position,improve,top_3,1.6,0.06,24784,0-30
7122,content_7a6df559322d,client_19581e27de,10,14.355110,underperforming_ctr_for_position,improve,top_3,0.7,0.14,43650,91-180


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. content_8451fc6f034d — improve. Ranks #2.3 (top_3) but only 0.03% CTR against a 1.48% peer
   average, across 272K impressions — huge visibility, barely any clicks. Trend is "up", so this
   isn't a dying page — a snippet/title fix could recover a lot of traffic. Would be wrong if: this
   query is branded/navigational and users are clicking a sitelink or knowledge panel instead of
   the blue link — not a real snippet problem.

2. content_8c19996aa890 — improve. #2.5 position, 0.15% CTR vs 1.48% peer average, 509K
   impressions — the single biggest reach in the top 10. Trend is "down". Would be wrong if: a
   SERP feature (featured snippet, PAA) above this result is absorbing clicks that no title rewrite
   can win back.

3. content_4a6607efcb46 — improve. #2.2, 0.01% CTR (the lowest in the top 10) vs 1.48% peer
   average, 128K impressions. Trend "up". Would be wrong if: this CTR is a data/tracking error
   rather than a real user behavior — worth spot-checking before acting.

4. content_e12868d1f396 — improve. #2.9, 0.07% CTR vs 1.48%, 150K impressions, trend "stable".
   Would be wrong if: this page's actual intent match is poor (ranks well on a keyword it doesn't
   really answer), in which case a snippet fix won't help — the content itself needs work.

5. content_8053a66bd6ac — improve. #2.6, 0.08% CTR vs 1.48%, 53K impressions, trend "down".
   Would be wrong if: the decline is seasonal and will self-correct without any snippet change.

6. content_d225ec9f3d46 — improve. #0.7 (best position in the top 10), 0.05% CTR vs 1.48%, 26K
   impressions, trend "down". Would be wrong if: position 0.7 reflects a rich result (like an
   image pack) that doesn't behave like a normal blue link — the peer-average comparison wouldn't
   be fair here.

7. content_0022a6b4290f — improve. #1.2, 0.07% CTR vs 1.48%, 30K impressions, trend "down".
   Would be wrong if: the drop tracks a broader site-wide traffic dip, not something specific to
   this page's snippet.

8. content_6f81ccd92b64 — improve. #2.9, 0.19% CTR (the highest CTR in this top 10, still far
   below peer average) vs 1.48%, 74K impressions, trend "stable". Would be wrong if: 0.19% is
   actually typical for this specific query type and the "peer average" is being pulled up by a
   few outlier pages in the same tier.

9. content_f4e210ee0c27 — improve. #1.6, 0.06% CTR vs 1.48%, 25K impressions, trend "down".
   Would be wrong if: this is a duplicate/near-duplicate of another ranking page cannibalizing its
   own clicks — the fix would be consolidation, not a snippet rewrite.

10. content_7a6df559322d — improve. #0.7, 0.14% CTR vs 1.48%, 44K impressions, trend "down".
    Would be wrong if: same as #6 — position 0.7 likely means a non-standard SERP feature, not a
    normal ranked result.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
out.head(10)

,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action_label,position_tier,avg_position,ctr,impressions_90d,freshness_tier
7678,content_8451fc6f034d,client_d029fa3a95,1,18.190613,underperforming_ctr_for_position,improve,top_3,2.3,0.03,272144,0-30
26844,content_8c19996aa890,client_4e07408562,2,17.524576,underperforming_ctr_for_position,improve,top_3,2.5,0.15,509252,0-30
3331,content_4a6607efcb46,client_6208ef0f77,3,17.330138,underperforming_ctr_for_position,improve,top_3,2.2,0.01,128068,91-180
16736,content_e12868d1f396,client_4e07408562,4,16.845255,underperforming_ctr_for_position,improve,top_3,2.9,0.07,149712,0-30
9412,content_8053a66bd6ac,client_19581e27de,5,15.260254,underperforming_ctr_for_position,improve,top_3,2.6,0.08,52687,91-180
7860,content_d225ec9f3d46,client_f369cb89fc,6,14.599610,underperforming_ctr_for_position,improve,top_3,0.7,0.05,26470,0-30
27107,content_0022a6b4290f,client_f369cb89fc,7,14.560919,underperforming_ctr_for_position,improve,top_3,1.2,0.07,29747,0-30
10893,content_6f81ccd92b64,client_19581e27de,8,14.498052,underperforming_ctr_for_position,improve,top_3,2.9,0.19,73675,91-180
20537,content_f4e210ee0c27,client_7f2253d7e2,9,14.404082,underperforming_ctr_for_position,improve,top_3,1.6,0.06,24784,0-30
7122,content_7a6df559322d,client_19581e27de,10,14.355110,underperforming_ctr_for_position,improve,top_3,0.7,0.14,43650,91-180


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak pick, honestly:** All 10 of my top picks are position_tier = top_3. That's not a
coincidence — top_3 has by far the highest peer average CTR (1.48%, versus 0.15–0.65% everywhere
else), so my formula can only produce a large gap there. A page ranked "striking" or "page_3_5"
with an equally bad *relative* CTR shortfall will never make this list, because the absolute gap
against its own (much lower) peer average is small. My rule is structurally blind to CTR problems
outside the very top of the results — worth fixing before Week 5 by normalizing the gap as a
percentage of the peer average instead of a raw difference.

**Leakage check:** My score formula uses only impressions_90d, ctr, and position_tier — all
observed within the same 90-day window, nothing from a different or future time period. I did not
use trend_direction, trend_pct, or is_declining_label anywhere in the score (they're absent from
the formula — only impressions_90d, ctr, and position_tier feed baseline_action_score). I also did
not use needs_ctr_fix, health_score, priority_score, or action_type — confirmed these aren't even
present in this dataset, so there was nothing to accidentally leak in.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Programmatic confirmation of the leakage check above
excluded_cols = ["trend_direction", "trend_pct", "is_declining_label",
                  "needs_ctr_fix", "health_score", "priority_score", "action_type"]
score_inputs = ["impressions_90d", "ctr", "position_tier"]
print("Score built only from:", score_inputs)
print("Excluded columns present in dataset:", [c for c in excluded_cols if c in df.columns])

Score built only from: ['impressions_90d', 'ctr', 'position_tier']
Excluded columns present in dataset: ['trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.